Corrido para angular 1º mask C1, em medium session.

### 1. Initialization

Setting up the name of the file that will have the alpha values and importing some global parameters and functions.

(Couldn't most of this be done in param_multi_mask.py, since it just takes parameters from there? It would be a lot cleaner)

In [70]:
## ----- IPYTHON COMMANDS ----- ##
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
import os
os.getcwd()

'/idia/users/bvitoria/satellite_RFI-mine/Notebooks/chi2_testing/chi2_alpha_optimization'

In [71]:
## ----- IMPORTS ----- ##
from satellite_RFI.src import tools  # <-- some useful functions
from satellite_RFI.src.satellite_sims import satellite_sim
import sys
sys.path.insert(0, './param_import/')  # <-- alters the path that searches for imports
from imports import *  # <-- file with the general imports (numpy, astropy, matplotlib, etc)
import time


## ----- GLOBAL PARAMETERS ----- ##
## observation name + date
import param_multi_mask as pm
fname, dtime = tools.timepoint(fname=pm.file, date=None)  # <-- observation name + date
file_loc = pm.data_save + pm.folder  # <-- file path
name = str(pm.file) + "_"  # <-- file name


Date of observation: 2019-02-25 02:40:11
Fname: 1551055211


In [33]:
data = pickle.load(open(pm.data_save 
                  + str(fname)
                  + "_individual_satellite_angular_position_"
                  + str(pm.beam_model)
                  + "_beam_"
                  + str(pm.fs)
                  + "_"
                  + str(pm.fe)
                  + "MHz" + ".p","rb",), encoding="latin1",)

In [21]:
print("data is a {}".format(type(data)))
print("data's keys are: {}".format(data.keys()))
print("each value is a {}".format(type(data["galileo"])))
print("and each constellation dictionary has this number of values: {}".format(
    [len(data[x].keys()) for x in data.keys()] ))

for x in data.keys():
    print("For key {} we have:".format(x))
    print(data[x].keys())
    print("First map:")
    print(type(data[x][list(data[x].keys())[0]]))

data is a <class 'dict'>
data's keys are: dict_keys(['gps-ops', 'galileo', 'irnss', 'sbas', 'glo-ops', 'beidou'])
each value is a <class 'dict'>
and each constellation dictionary has this number of values: [15, 13, 5, 9, 13, 18]
For key gps-ops we have:
dict_keys([b'GPS-BIIF-7(PRN-09)', b'GPS-BIIF-12-(PRN-32)', b'GPS-BIIF-8(PRN-03)', b'GPS-BIIF-9(PRN-26)', b'GPS-BIIF-10-(PRN-08)', b'GPS-BIIR-6(PRN-14)', b'GPS-BIIR-3(PRN-11)', b'GPS-BIIA-23-(PRN-18)', b'GPS-BIIR-8(PRN-16)', b'GPS-BIIRM-2-(PRN-31)', b'GPS-BIIF-4(PRN-27)', b'GPS-BIIRM-6-(PRN-07)', b'GPS-BIIR-10-(PRN-22)', b'GPS-BIIR-9(PRN-21)', b'GPS-BIIR-12-(PRN-23)'])
First map:
<class 'numpy.ma.core.MaskedArray'>
For key galileo we have:
dict_keys([b'GSAT0201-(PRN-E18)', b'GSAT0205-(PRN-E24)', b'GSAT0209-(PRN-E09)', b'GSAT0210-(PRN-E01)', b'GSAT0222-(PRN-E33)', b'GSAT0213-(PRN-E04)', b'GSAT0216-(PRN-E25)', b'GSAT0203-(PRN-E26)', b'GSAT0214-(PRN-E05)', b'GSAT0102-(PRN-E12)', b'GSAT0212-(PRN-E03)', b'GSAT0218-(PRN-E31)', b'GSAT0101-(PRN-

In [13]:
## ----- FREQUENCY RANGE ----- ##
f_write = []
for f in [pm.fs_slice,pm.fe_slice]:
    if f is None:  f_write.append("inf")
    else:  f_write.append(str(f))
print("Frequency range: {} - {} MHz".format(*f_write))
freq_name = str(pm.fs_slice) + "-" + str(pm.fe_slice) + "_"


Frequency range: 1100 - 1350 MHz


In [14]:
## ----- TIME RANGE ----- ##
t_write = []
t_name = []
for i,t in enumerate([pm.ts_slice,pm.te_slice]):
    if t is None:  
        t_write.append("inf")
        t_name.append(str(np.round(pm.nd_s0[-i], 2)))
    else:  
        t_write.append(str(t))
        t_name.append(str(t))
print("Time range: {} - {} seconds".format(*t_write))
time_name = t_name[0] + "-" + t_name[1] + "_"

## ----- TIME AVERAGING ----- ##
if pm.time_average is not None:  time_average_name = "time_average_" + str(pm.time_average) + "_"
else:  time_average_name = ""


Time range: inf - inf seconds


In [15]:
## ----- DAMPER ----- ##
if pm.dampner is None:
    print("No dampening.")
    dampner_name = "no-dampening_"

elif pm.dampner == "gaussian":
    print("Gaussian Out of Band dampening given.")
    
    if pm.dampner_sigma is None:
        print("Random dampner values given.")
        dampner_name = pm.dampner + "-random_"
        
    else:
        print("Damper values fixed to " + str(pm.dampner_sigma) + " level.")
        dampner_name = pm.dampner + "-" + str(pm.dampner_sigma) + "_"


No dampening.


In [16]:
## ----- CHI-SQUARE SIGMA ----- ##
print("The Chi-Square denominator will be: ",end="")
if pm.chi_sigma == True:
    print("radiometer equation.")
    chi_sig_name = "fraction_"
elif pm.chi_sigma == False:
    print("1.")
    chi_sig_name = "residual_"
else:
    print("Error!")


The Chi-Square denominator will be: radiometer equation.


In [17]:
## ----- FINAL FILE NAME ----- ##
file_save = (file_loc
    + name
    + freq_name
    + time_name
    + pm.mask_name
    + dampner_name
    + chi_sig_name
    + time_average_name
    + pm.save_suffix
)
print("File location is: \n" + file_save)


File location is: 
/idia/projects/hi_im/satellite_rfi/Testing/1551055211/sat_12/1551055211_1100-1350_774.75-6474.34_no-mask_no-dampening_fraction_iara


### 2. Calculating alpha values

In [33]:
##====================================================================================================================================##
## INITIALIZING THE FUNCTION
sat = satellite_sim(
    file_name=fname,
    sats_only=False,
    data_loc=pm.data_save,
    sat_loc=pm.data_save,
    survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency],
    sat_info=pm.path_catalog,
    plots_loc=pm.data_plot,
    sat_beam=pm.beam_model + "_beam_" + str(pm.fs) + "_" + str(pm.fe) + "MHz",
    frequency_range=[pm.fs, pm.fe],
    constellations=pm.constellations_remain,
    nearby_satellites=pm.nearby_constellations,
    verbose=False,
)


['gps-ops', 'glo-ops', 'galileo', 'beidou', 'irnss', 'sbas']
['gps-ops', 'glo-ops', 'galileo', 'beidou', 'irnss', 'sbas']


In [29]:
## INTIAL ALPHA DICTIONARY VALUES
## NOTE - There can be a length issue, the length should be the same as number of signals available
alpha_dic = np.zeros(pm.no_signals)

## INTIAL SIGMA DICTIONARY VALUES
if pm.dampner is not None:
    # Note- There can be a length issue here, be aware
    if pm.dampner_sigma is not None:
        print("Dampner will be fixed.")
        sigma_dic = pm.dampner_sigma * np.ones(21)
        # The input values for the Optimization of the Chi-Sqaure fitting
        input_param = alpha_dic
    else:
        print("Dampner will be random.")
        sigma_dic = np.ones(21)
        input_param = np.concatenate((alpha_dic, sigma_dic))

else:
    sigma_dic = None
    input_param = alpha_dic


In [41]:
#====================================================================================================================================##
## EXCECUTING THE INITIAL FUNCTION
sat.excecute(
    a_param=alpha_dic,  # Should always check this value
    obs_time_start=pm.ts_slice,
    obs_time_end=pm.te_slice,
    obs_frequency_start=pm.fs_slice,
    obs_frequency_end=pm.fe_slice,
    file_bias_choice=pm.bias,
    add_sub=[True, False],
    attenuation_func=pm.dampner,
    attenuation_sigma=sigma_dic,
    bandsize=None,
    verbose=True,
)


Number of signals inside choice frequency range:  21


In [42]:
##====================================================================================================================================##
## CHI SQAURE METHOD
def chisq_func2(params):
    ## FITTING PARAMETERS
    a_param = params[: pm.no_signals]
    if pm.dampner is not None:
        if pm.dampner_sigma is None:
            # If it is random then the sigma will work
            s_param = params[21:]  # <-- não percebo, são parâmetros adicionais??????
            # If it is not random then the sigmai will fixed
        elif pm.dampner_sigma is not None:
            s_param = pm.dampner_sigma * np.ones(21)
        else:
            print("Error in sigma fitting parameters.")
    else:
        s_param = None

    ## EXCECUTION
    sat.excecute(
        a_param=a_param,
        obs_time_start=pm.ts_slice,
        obs_time_end=pm.te_slice,
        obs_frequency_start=pm.fs_slice,
        obs_frequency_end=pm.fe_slice,
        file_bias_choice=pm.bias,
        add_sub=[True, False],
        attenuation_func=pm.dampner,
        attenuation_sigma=s_param,
        bandsize=None,
        verbose=False,
    )

    ##====================================================================================================================================##
    ## MASK TYPE
    ## No mask
    if (
        pm.mask_degree is None
        and pm.mask_temperature is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=None)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=None)

    ## Angular mask
    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        simulation = np.ma.array(
            data=sat.simulation_TOD_slice.T, mask=sat.mask_nearby_satellites_slice
        )
        data = np.ma.array(
            data=sat.calibration_data_slice.T, mask=sat.mask_nearby_satellites_slice
        )

    ## Temperature mask
    elif (
        pm.mask_temperature is not None
        and pm.mask_degree is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        max_k = pm.mask_temperature
        zero_arr = np.zeros(sat.calibration_data_slice.shape)
        mask_idx = np.where(sat.calibration_data_slice > max_k)
        zero_arr[mask_idx] = 1
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)

    ## Temporal mask
    elif (
        pm.mask_degree is None
        and pm.mask_temperature is None
        and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
        and pm.mask_pixel_timeline is None
    ):
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=None)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=None)
        
    ## Pixel timeline mask
    elif (
            pm.mask_degree is None
            and pm.mask_temperature is None
            and pm.mask_temporal[0] is None
            and pm.mask_temporal[1] is None
            and pm.mask_pixel_timeline is not None
        ):
        masker_tod = tools.time_line_masker(data_in = sat.calibration_data_slice.T, t_value = pm.mask_pixel_timeline)
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=masker_tod)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=masker_tod)


    ## Angular+Thermal mask
    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is not None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        max_k = pm.mask_temperature
        zero_arr = np.zeros(sat.calibration_data_slice.shape)
        mask_idx = np.where(sat.calibration_data_slice > max_k)
        zero_arr[mask_idx] = 1
        sim = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)
        dat = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)
        simulation = np.ma.array(data=sim, mask=sat.mask_nearby_satellites_slice)
        data = np.ma.array(data=dat, mask=sat.mask_nearby_satellites_slice)

    ## Angular+Temporal mask
    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is None
        and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
        and pm.mask_pixel_timeline is None
    ):
        simulation = np.ma.array(
            data=sat.simulation_TOD_slice.T, mask=sat.mask_nearby_satellites_slice
        )
        data = np.ma.array(
            data=sat.calibration_data_slice.T, mask=sat.mask_nearby_satellites_slice
        )

    ## Thermal+Temporal mask
    elif (
        pm.mask_degree is None
        and pm.mask_temperature is not None
        and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
        and pm.mask_pixel_timeline is None
    ):
        max_k = pm.mask_temperature
        zero_arr = np.zeros(sat.calibration_data_slice.shape)
        mask_idx = np.where(sat.calibration_data_slice > max_k)
        zero_arr[mask_idx] = 1
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)

    ## Angular+Thermal+Temporal
    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is not None
        and pm.mask_temporal[0] is not None
        and pm.mask_temporal[1] is not None
        and pm.mask_pixel_timeline is None
    ):
        max_k = pm.mask_temperature
        zero_arr = np.zeros(sat.calibration_data_slice.shape)
        mask_idx = np.where(sat.calibration_data_slice > max_k)
        zero_arr[mask_idx] = 1
        sim = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)
        dat = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)
        simulation = np.ma.array(data=sim, mask=sat.mask_nearby_satellites_slice)
        data = np.ma.array(data=dat, mask=sat.mask_nearby_satellites_slice)

    else:
        print("Error in masking choice")

    ##====================================================================================================================================##

    ## TIME AVERAGING
    if pm.time_average is not None:
        data = tools.waterfall_avg_time(
            timer=pm.nd_s0[sat.time_idx[0] : sat.time_idx[1]],
            size=pm.time_average,
            waterfall=data,
        )
        simulation = tools.waterfall_avg_time(
            timer=pm.nd_s0[sat.time_idx[0] : sat.time_idx[1]],
            size=pm.time_average,
            waterfall=simulation,
        )

    ##====================================================================================================================================##

    ## CHI SQUARE EQUATION
    ## Denominator value
    if pm.chi_sigma == True:
        sig = data
    elif pm.chi_sigma == False:
        sig = 1
    else:
        print("Error in sigma value for Chi-Square")
    ## Numerator value
    chi_num = simulation - data
    ## Chi-Square
    chi_sq = np.ma.sum(chi_num ** 2 / sig ** 2)

    ##====================================================================================================================================##
    print(chi_sq)
    return chi_sq


In [43]:
test = chisq_func2(params=alpha_dic)  # <-- está errado!

305458787.47577375


In [ ]:
##====================================================================================================================================##
## PRIOR-BOUNDARY INFORMATION
## Boundary values
bnd_val = (0.0, None)
print("Boundary values are set at: ", bnd_val)
## Seeting bounds for the Chi-Square
if pm.dampner is not None:
    if pm.dampner_sigma is None:
        bnds = [bnd_val for bnd_i in range(2 * sat.alpha_len)]
    elif pm.dampner_sigma is not None:
        bnds = [bnd_val for bnd_i in range(sat.alpha_len)]
    else:
        print("Error in setting in boundary conditons.")
else:
    bnds = [bnd_val for bnd_i in range(sat.alpha_len)]


### 3. Running the optimization

In [ ]:
##====================================================================================================================================##
## RUNNING THE OPTIMIZATION PARAMETERS
print("Running optimization")
start = time.perf_counter()
signal_PL = opt.minimize(
    fun = chisq_func2,
    x0 = input_param,  # Set to none becuase the number of signals will be determined by the bandsize
    method = "Powell",
    bounds = bnds,
    tol = 1e-6,
    options={"maxiter": 20},
)
elapsed = time.perf_counter() - start
print(f"This took {elapsed} seconds, or {elapsed/(60*60)} hours".format())


In [ ]:
##====================================================================================================================================##
## BEST FIT VALUES
if pm.dampner is not None:
    # Note- There can be a length issue here, be aware
    if pm.dampner_sigma is not None:
        print("Dampner will be fixed.")
        sigma_param_bf = pm.dampner_sigma * np.ones(21)
        # The input values for the Optimization of the Chi-Sqaure fitting
        a_param_bf = signal_PL.x
    else:
        print("Dampner was best fitted.")
        sigma_param_bf = signal_PL.x[21:]
        a_param_bf = signal_PL.x[:21]

else:
    sigma_param_bf = None
    a_param_bf = signal_PL.x

In [ ]:
##====================================================================================================================================##
## RUNNING SECOND INITIALIZATION
print("Running 2nd init")
sat2 = satellite_sim(
    file_name=fname,
    sats_only=False,
    data_loc=pm.data_save,
    sat_loc=pm.data_save,
    survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency],
    path_catalog=pm.path_catalog,
    plots_loc=pm.data_plot,
    sat_beam=pm.beam_model + "_beam_" + str(pm.fs) + "_" + str(pm.fe) + "MHz",
    frequency_range=[pm.fs, pm.fe],
    constellations=pm.constellations_remain,
    nearby_satellites=pm.nearby_constellations,
    verbose=False,
)

sat2.excecute(
    a_param=a_param_bf,
    obs_time_start=pm.ts_slice,
    obs_time_end=pm.te_slice,
    obs_frequency_start=pm.fs_slice,
    obs_frequency_end=pm.fe_slice,
    file_bias_choice=pm.bias,
    add_sub=[True, False],
    attenuation_func=pm.dampner,
    attenuation_sigma=sigma_param_bf,
    bandsize=None,
    verbose=False,
)


In [ ]:
##====================================================================================================================================##
## SAVING DATA INFORMATION
data_info = {
    "initial" : input_param,
    "time" : [pm.ts_slice, pm.te_slice],
    "frequency_slice" : [pm.fs_slice, pm.fe_slice],
    "best-fit" : signal_PL.x,
    "chi2_value" : signal_PL.fun,
    "chi2_div" : signal_PL.fun / sat2.simulation_TOD_slice.size,
}

print("Saving file {}".format(pm.filename))
pickle.dump(data_info, open(pm.filename + ".p", "wb"))
